# 02 — Data Preprocessing

**Goal:** convert the untouched transaction data into a reproducible cleaned transaction dataset.

Every removal or transformation is documented and counted. The raw Excel file is never overwritten.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
# When notebooks are launched from notebooks/, move to repository root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "online_retail_II.xlsx"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"
MODELS = PROJECT_ROOT / "models"

DATA_PROCESSED.mkdir(exist_ok=True, parents=True)
FIGURES.mkdir(exist_ok=True, parents=True)
MODELS.mkdir(exist_ok=True, parents=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

xl = pd.ExcelFile(DATA_RAW)
df = pd.concat(
    [pd.read_excel(DATA_RAW, sheet_name=s).assign(SourceSheet=s) for s in xl.sheet_names],
    ignore_index=True
)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

print("Raw shape:", df.shape)

## 1. Keep a cleaning log

In [ ]:
def log_stage(log, name, frame, note):
    log.append({
        "stage": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "note": note
    })

clean_log = []
log_stage(clean_log, "Raw combined", df, "Original rows from both Online Retail II sheets")

## 2. Remove exact duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates().copy()
removed = before - len(df)
print(f"Exact duplicates removed: {removed}")
log_stage(clean_log, "After duplicate removal", df, f"Removed {removed} exact duplicate rows")

## 3. Standardize types and derive transaction value

In [ ]:
df["Invoice"] = df["Invoice"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df["Country"] = df["Country"].astype(str).str.strip()
df["Description"] = df["Description"].astype("string").str.strip()

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Customer ID"] = pd.to_numeric(df["Customer ID"], errors="coerce")

# Transaction value is useful for later feature engineering.
df["TotalPrice"] = df["Quantity"] * df["Price"]

print(df[["Quantity", "Price", "TotalPrice"]].dtypes)

## 4. Handle invalid dates

In [ ]:
before = len(df)
df = df.dropna(subset=["InvoiceDate"]).copy()
removed = before - len(df)
print(f"Rows with invalid/missing InvoiceDate removed: {removed}")
log_stage(clean_log, "After invalid-date handling", df, f"Removed {removed} rows with invalid/missing dates")

## 5. Separate customer-identifiable modeling transactions

In [ ]:
# CustomerID cannot be safely imputed because the target is customer-level.
known_customer = df["Customer ID"].notna()

print("Rows with Customer ID:", int(known_customer.sum()))
print("Rows without Customer ID:", int((~known_customer).sum()))

# We keep the full cleaned transaction table at this stage.
# A customer-identifiable subset is created for the modeling pipeline below.
customer_df = df.loc[known_customer].copy()

## 6. Handle cancellation/return and invalid-price records for purchase modeling

In [ ]:
customer_df["IsCancellation"] = (
    customer_df["Invoice"].str.startswith("C") |
    (customer_df["Quantity"] < 0)
)

print("Cancellation/return records:", int(customer_df["IsCancellation"].sum()))
print("Invalid/non-positive prices:", int((customer_df["Price"] <= 0).sum()))

# Main purchase-history features will use normal positive-value purchases.
purchase_df = customer_df.loc[
    (~customer_df["IsCancellation"]) &
    (customer_df["Quantity"] > 0) &
    (customer_df["Price"] > 0) &
    (customer_df["TotalPrice"] > 0)
].copy()

log_stage(
    clean_log,
    "Purchase-only modeling transactions",
    purchase_df,
    "Customer identified; non-cancellation; positive quantity; positive price"
)
print("Purchase modeling rows:", len(purchase_df))

## 7. Outlier analysis

In [ ]:
# Flag, rather than blindly delete, statistical outliers.
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr

outlier_rows = []
for col in ["Quantity", "Price", "TotalPrice"]:
    lo, hi = iqr_bounds(purchase_df[col])
    flagged = ((purchase_df[col] < lo) | (purchase_df[col] > hi)).sum()
    outlier_rows.append({
        "feature": col,
        "lower_bound": lo,
        "upper_bound": hi,
        "flagged_count": int(flagged),
        "flagged_pct": round(flagged / len(purchase_df) * 100, 2)
    })

outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

# Do not delete legitimate large purchases solely because IQR flags them.

## 8. Transformation candidates

In [ ]:
# Inspect skew before deciding on log transformations.
skew_report = purchase_df[["Quantity", "Price", "TotalPrice"]].skew().sort_values(ascending=False)
display(skew_report.to_frame("skewness"))

## 9. Save cleaned transaction datasets

In [ ]:
clean_path = DATA_PROCESSED / "clean_transactions.csv"
purchase_path = DATA_PROCESSED / "purchase_transactions.csv"
log_path = DATA_PROCESSED / "preprocessing_log.csv"

# clean_transactions keeps customer-unidentified rows for audit continuity.
df.to_csv(clean_path, index=False)
# purchase_transactions is the main transaction set used to build customer histories.
purchase_df.to_csv(purchase_path, index=False)
pd.DataFrame(clean_log).to_csv(log_path, index=False)

display(pd.DataFrame(clean_log))
print("Saved:", clean_path)
print("Saved:", purchase_path)
print("Saved:", log_path)

## 10. Before/after evidence

In [ ]:
summary = pd.DataFrame(clean_log)
display(summary)